# Dataset Analysis

Рабочий ноутбук для анализа накопленного `synthesized_pii.jsonl`: общая статистика, exact mismatches, похожие пары и быстрый просмотр конкретных примеров.

Запускать из папки `synthetic_data_generation/synthesizer_agent/notebooks`. Ноутбук ничего не перезаписывает и не чистит, только читает JSONL.

In [56]:
from pathlib import Path
import json
import sys
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from metrics_eval import load_jsonl, compute_all_metrics, _normalize_for_similarity

DATASET_PATH = Path("../outputs/synthesized_pii.jsonl")
results = load_jsonl(DATASET_PATH)

print(f"Dataset: {DATASET_PATH.resolve()}")
print(f"Loaded texts: {len(results)}")

Dataset: /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/synthesized_pii.jsonl
Loaded texts: 1825


## Metrics Summary

Основной компактный отчет. `exact_mismatches` и `nearest_similarity_values` отдельно не печатаем целиком, чтобы не забивать вывод.

In [57]:
report = compute_all_metrics(results)

summary = {
    "dataset_stats": report["dataset_stats"],
    "tag_correctness": {
        k: v for k, v in report["tag_correctness"].items()
        if k != "exact_mismatches"
    },
    "semantic_repetition": {
        k: v for k, v in report["semantic_repetition"].items()
        if k != "nearest_similarity_values"
    },
}

pprint(summary, sort_dicts=False)

{'dataset_stats': {'total_texts': 1825,
                   'total_entities': 2779,
                   'texts_with_entities': 1825,
                   'texts_without_entities': 0,
                   'entities_per_text': {'min': 1,
                                         'max': 3,
                                         'mean': 1.5227397260273972,
                                         'median': 2.0,
                                         'mode': 2,
                                         'distribution': {1: 891,
                                                          2: 914,
                                                          3: 20}},
                   'entity_type_counts': {'ADDRESS': 305,
                                          'BANK_CARD': 310,
                                          'EMAIL': 310,
                                          'NAME': 309,
                                          'ORGANIZATION': 309,
                                          'PASSPORT

## Entity Counts

Сколько примеров набрано по каждой сущности. Это удобно смотреть, когда добиваем условные `300+` на класс.

In [58]:
entity_counts_df = (
    pd.DataFrame(
        sorted(report["dataset_stats"]["entity_type_counts"].items()),
        columns=["entity", "count"],
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

entity_counts_df

,entity,count
0,BANK_CARD,310
1,EMAIL,310
2,PASSPORT_RF,310
3,NAME,309
4,ORGANIZATION,309
5,PHONE_NUMBER,309
6,VK,309
7,TELEGRAM,308
8,ADDRESS,305


## Entities Per Text

Распределение числа сущностей на один синтетический текст.

In [43]:
entities_per_text_df = pd.DataFrame(
    sorted(report["dataset_stats"]["entities_per_text"]["distribution"].items()),
    columns=["entities_per_text", "num_texts"],
)

entities_per_text_df

,entities_per_text,num_texts
0,1,894
1,2,914
2,3,20


## Exact Mismatches

Это диагностическая проверка: `used_entities.value` должен буквально совпасть с содержимым тега.

Для `NAME`, `ORGANIZATION`, `ADDRESS` расхождения часто могут быть нормальными склонениями или нормализацией адреса, поэтому это не всегда ошибка датасета. Для регулярочных сущностей вроде `EMAIL`, `PHONE_NUMBER`, `BANK_CARD`, `VK`, `TELEGRAM`, `PASSPORT_RF` расхождения обычно подозрительнее.

In [44]:
mismatches = report["tag_correctness"]["exact_mismatches"]
print(f"Exact mismatches: {len(mismatches)}")

mismatch_rows = []
for m in mismatches:
    mismatch_rows.append({
        "jsonl_line": m["jsonl_line"],
        "entity_index": m["entity_index"],
        "key": m["key"],
        "value": m["value"],
        "actual_tagged_values": " | ".join(m["actual_tagged_values"]),
    })

mismatches_df = pd.DataFrame(mismatch_rows)
mismatches_df.head(5)

Exact mismatches: 70


,jsonl_line,entity_index,key,value,actual_tagged_values
0,365,1,ORGANIZATION,Элегия,Элегии
1,422,0,NAME,Евдокия Ивановна МАКСИМОВА,Евдокию Ивановну МАКСИМОВУ
2,422,1,ADDRESS,Колпашево наб проезжая 158 кв 157,"Колпашево, набережная Проезжая, 158, кв. 157"
3,430,1,ADDRESS,Озерный бул. 128 кв 228,"Озерном бульваре, 128, квартира 228"
4,431,1,ADDRESS,"Рыбинск, Набережная Санаторная 144 кв 25","Рыбинске, Набережная Санаторная 144 кв 25"


In [45]:
for i, row in mismatches_df.iterrows():
    print('idx:', i, 'value:', row['value'], 'actual_tagged_value:', row['actual_tagged_values'])

idx: 0 value: Элегия actual_tagged_value: Элегии
idx: 1 value: Евдокия Ивановна МАКСИМОВА actual_tagged_value: Евдокию Ивановну МАКСИМОВУ
idx: 2 value: Колпашево наб проезжая 158 кв 157 actual_tagged_value: Колпашево, набережная Проезжая, 158, кв. 157
idx: 3 value: Озерный бул. 128 кв 228 actual_tagged_value: Озерном бульваре, 128, квартира 228
idx: 4 value: Рыбинск, Набережная Санаторная 144 кв 25 actual_tagged_value: Рыбинске, Набережная Санаторная 144 кв 25
idx: 5 value: Брагин Гостомысл М. actual_tagged_value: Брагина Гостомысла М.
idx: 6 value: Завод Сельмаш actual_tagged_value: Завода Сельмаш
idx: 7 value: Малая ул. 20/4 кв 64 actual_tagged_value: Малой ул. 20/4 кв 64
idx: 8 value: Газстрой actual_tagged_value: Газстрое
idx: 9 value: Карелия, Кижи, Санаторный пр. 100/2/200 actual_tagged_value: Карелии, на Кижах, Санаторный проезд 100/2/200
idx: 10 value: Нефтеюганск, пер. Донской 51 кв 11 actual_tagged_value: Нефтеюганске, пер. Донской 51 кв 11
idx: 11 value: Таганрог, пер. Белор

In [46]:
if mismatches_df.empty:
    print("Exact mismatches не найдены.")
else:
    display(mismatches_df.groupby("key").size().sort_values(ascending=False).to_frame("count"))

,count
key,
ADDRESS,33
ORGANIZATION,20
NAME,12
PASSPORT_RF,5


In [53]:
def show_mismatch(i: int):
    """Показать один mismatch по номеру строки в mismatches, не по jsonl_line."""
    m = mismatches[i]
    print("=" * 100)
    print(f"Mismatch #{i}")
    print(f"JSONL line: {m['jsonl_line']}")
    print(f"Entity: {m['key']}")
    print(f"Expected value: {m['value']!r}")
    print(f"Actual tagged values: {m['actual_tagged_values']}")
    print("-" * 100)
    print(m["text"])
    print("-" * 100)
    print(json.dumps(m["used_entities"], ensure_ascii=False, indent=2))

# if mismatches:
#     show_mismatch(0)

# suspect_mismatch_indexes = [
#     14,  # Новокузнецк ... -> в теге только улица/кв, город мог быть вне тега
#     20,  # березники ... -> в теге только Веселая наб..., город потерялся
#     23,  # Чита ... -> в теге только улица/дом, город потерялся
#     36,  # Баргузин, шоссе Шаумяна 70 -> Баргузин, дом 70, улица потерялась
#     38,  # паспорт серия... -> серия..., слово паспорт вне тега/потеряно
#     58,  # серия 0794,номер -> серия 0794, номер, мелочь, но для паспорта лучше глянуть
#     62,  # паспорт сер... -> паспортом сер..., грамматически ок, но для regex может быть спорно
#     64,  # Пугачевка ... -> Пугачевку..., странное склонение города
#     68,  # паспорт -> паспорту, грамматически ок, но для будущего regex/NER стоит глянуть
#     72,  # паспорт сер... -> паспорта сер..., тоже паспорт в падеже
# ]

# for idx in suspect_mismatch_indexes:
#     show_mismatch(idx)

## Similar Text Pairs

Поиск ближайших похожих пар тем же способом, что и в `metrics_eval.py`: TF-IDF по символьным n-граммам после замены всех PII-тегов на `[PII]`.

`pairs_df` показывает ближайшего соседа для каждого текста. Для ручной проверки удобнее начинать с верхних строк.

In [51]:
def build_nearest_pairs(results):
    texts_norm = [_normalize_for_similarity(item["text"]) for item in results]

    if len(texts_norm) <= 1:
        return pd.DataFrame(columns=[
            "text_index", "jsonl_line", "nearest_index", "nearest_jsonl_line", "similarity",
        ])

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    matrix = vectorizer.fit_transform(texts_norm)
    sim = cosine_similarity(matrix)
    np.fill_diagonal(sim, -1.0)

    pairs = []
    for i in range(sim.shape[0]):
        j = int(sim[i].argmax())
        pairs.append({
            "text_index": i,
            "jsonl_line": i + 1,
            "nearest_index": j,
            "nearest_jsonl_line": j + 1,
            "similarity": float(sim[i, j]),
        })

    return pd.DataFrame(pairs).sort_values("similarity", ascending=False).reset_index(drop=True)

pairs_df = build_nearest_pairs(results)
pairs_df.head(30)

,text_index,jsonl_line,nearest_index,nearest_jsonl_line,similarity
0,175,176,1365,1366,0.951962
1,1365,1366,175,176,0.951962
2,405,406,1365,1366,0.917198
3,123,124,57,58,0.913676
4,57,58,123,124,0.913676
5,1731,1732,170,171,0.905625
6,170,171,1731,1732,0.905625
7,890,891,1309,1310,0.905564
8,1309,1310,890,891,0.905564
9,520,521,866,867,0.896062


In [54]:
def show_pair(row_idx: int):
    """Показать пару из pairs_df по номеру строки в pairs_df."""
    row = pairs_df.iloc[row_idx]
    i = int(row["text_index"])
    j = int(row["nearest_index"])

    print("=" * 100)
    print(f"Pair #{row_idx}: lines {i + 1} and {j + 1}, similarity={row['similarity']:.4f}")
    print("-" * 100)
    print(results[i]["text"])
    print("-" * 100)
    print(results[j]["text"])

# if len(pairs_df):
#     show_pair(0)
suspect_pair_rows = [0, 2, 3, 5, 7]
for i in suspect_pair_rows:
    show_pair(i)

Pair #0: lines 176 and 1366, similarity=0.9520
----------------------------------------------------------------------------------------------------
я думаю скоро начну читать новеллу Аватара Короля. Уже зарегистрировался на сайте, вот даже email ввел <EMAIL>nynpfcxb1471@corp.example.net</EMAIL>... Ой, ну ладно, не важно <emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji>
----------------------------------------------------------------------------------------------------
я думаю скоро начну читать новвелу Аватара Короля. <emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji><emoji>Face with hand over mouth</emoji> Кстати, если кто-то уже читал или хочет обсудить — пишите, вот мой профиль <VK>vk.me/nfccyh5zh8gq</VK>.
Pair #2: lines 406 and 1366, similarity=0.9172
-----------------------------------------------------

## High Similarity Filter

Отдельная таблица только по подозрительным ближайшим парам. Порог можно менять: для жесткой проверки ставь `0.90`, для просмотра пограничных повторов можно временно снизить до `0.75-0.80`.

In [14]:
SIMILARITY_THRESHOLD = 0.90

high_similarity_df = pairs_df[pairs_df["similarity"] >= SIMILARITY_THRESHOLD].copy()
print(f"Pairs with nearest similarity >= {SIMILARITY_THRESHOLD}: {len(high_similarity_df)}")
high_similarity_df.head(50)

## Quick Text Viewer

Быстрый просмотр любого объекта по нулевому индексу Python. Если смотришь строку JSONL из метрик, используй `show_example(jsonl_line - 1)`.

In [15]:
def show_example(i: int):
    item = results[i]
    print("=" * 100)
    print(f"EXAMPLE #{i}, JSONL line {i + 1}")
    print("\nTEXT:\n")
    print(item.get("text", ""))
    print("\nLOGIC OF ENTRY:\n")
    print(item.get("logic_of_entry", ""))
    print("\nUSED ENTITIES:\n")
    print(json.dumps(item.get("used_entities", []), ensure_ascii=False, indent=2))
    print("\nPOSITION HINT:\n")
    print(item.get("position_hint", ""))
    print("\nSOURCE FRAGMENT:\n")
    print(item.get("source_fragment", ""))

show_example(0)

## Drop Selected Rows

Безопасная ячейка для удаления выбранных строк из накопительного JSONL по `jsonl_line`.


In [55]:
# Drop selected JSONL rows by jsonl_line.
# ВАЖНО: jsonl_line начинается с 1, как в mismatches_df.
# По умолчанию список пустой, поэтому ячейка ничего не удаляет.

from datetime import datetime


DROP_JSONL_LINES = [
    1366, 1732, 1310
]

if not DROP_JSONL_LINES:
    print("DROP_JSONL_LINES пустой: ничего не удаляю.")
else:
    drop_indexes = {line - 1 for line in DROP_JSONL_LINES}
    lines = DATASET_PATH.read_text(encoding="utf-8").splitlines()

    unknown_lines = [line for line in DROP_JSONL_LINES if line < 1 or line > len(lines)]
    if unknown_lines:
        raise ValueError(f"jsonl_line вне диапазона 1..{len(lines)}: {unknown_lines}")

    backup_path = DATASET_PATH.with_suffix(
        f".backup_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jsonl"
    )
    backup_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    kept_lines = [line for i, line in enumerate(lines) if i not in drop_indexes]
    DATASET_PATH.write_text("\n".join(kept_lines) + "\n", encoding="utf-8")

    print(f"Было строк: {len(lines)}")
    print(f"Удалено строк: {len(lines) - len(kept_lines)}")
    print(f"Осталось строк: {len(kept_lines)}")
    print(f"Backup: {backup_path.resolve()}")


Было строк: 1828
Удалено строк: 3
Осталось строк: 1825
Backup: /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/synthesized_pii.backup_2026-05-03_00-47-42.jsonl
